In [1]:
import os
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path(
    "/content/HACE_Financial_Sentiment_Analysis"
)

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Project:", PROJECT_ROOT)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Project: /content/HACE_Financial_Sentiment_Analysis
Device: cuda
GPU: Tesla T4


In [2]:
# ============================================================
# CELL 2 — GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

DRIVE_MODEL_DIR = Path(
    "/content/drive/MyDrive/HACE_models"
)

print("Model directory:", DRIVE_MODEL_DIR)
print("Exists:", DRIVE_MODEL_DIR.exists())

if DRIVE_MODEL_DIR.exists():
    print("\nSaved models:")
    for item in sorted(DRIVE_MODEL_DIR.iterdir()):
        if item.is_dir():
            print("✓", item.name)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model directory: /content/drive/MyDrive/HACE_models
Exists: True

Saved models:
✓ finance_news
✓ fiqa
✓ phrasebank
✓ twitter


In [3]:
# ============================================================
# CELL 3 — HACE COMPONENTS
# ============================================================

from src.models.predictor import (
    ExpertPredictor,
    EXPERT_KEYS
)

from src.ensemble.feature_fusion import (
    FeatureFusion
)

from src.ensemble.meta_learner import (
    MetaLearner
)

from src.hedging.detector import (
    HedgeDetector
)


# ------------------------------------------------------------
# 1. Expert configuration
# ------------------------------------------------------------

print("=" * 60)
print("HACE COMPONENT CHECK")
print("=" * 60)

print("\nExpert order:")
print(EXPERT_KEYS)


# ------------------------------------------------------------
# 2. Expert Predictor
# ------------------------------------------------------------

predictor = ExpertPredictor(
    models_dir=DRIVE_MODEL_DIR,
    device=DEVICE
)

print("\n✓ ExpertPredictor initialized")


# ------------------------------------------------------------
# 3. Base Ensemble Feature Fusion
# ------------------------------------------------------------

base_fusion = FeatureFusion(
    use_hedge_features=False
)

print("\n✓ Base FeatureFusion initialized")
print("Base feature count:", len(base_fusion.feature_names))

print("Base features:")
for i, name in enumerate(
    base_fusion.feature_names
):
    print(f"{i:2d}: {name}")


# ------------------------------------------------------------
# 4. HACE Feature Fusion
# ------------------------------------------------------------

hace_fusion = FeatureFusion(
    use_hedge_features=True
)

print("\n✓ HACE FeatureFusion initialized")
print("HACE feature count:", len(hace_fusion.feature_names))

print("HACE features:")
for i, name in enumerate(
    hace_fusion.feature_names
):
    print(f"{i:2d}: {name}")


# ------------------------------------------------------------
# 5. Base Meta-Learner
# ------------------------------------------------------------

base_meta = MetaLearner(
    use_hedge_features=False,
    C=1.0,
    max_iter=1000,
    seed=SEED
)

print("\n✓ Base MetaLearner initialized")


# ------------------------------------------------------------
# 6. HACE Meta-Learner
# ------------------------------------------------------------

hace_meta = MetaLearner(
    use_hedge_features=True,
    C=1.0,
    max_iter=1000,
    seed=SEED
)

print("✓ HACE MetaLearner initialized")


# ------------------------------------------------------------
# 7. Hedging Detector
# ------------------------------------------------------------

hedge_detector = HedgeDetector()

print("✓ HedgeDetector initialized")


print("\n" + "=" * 60)
print("ALL HACE COMPONENTS INITIALIZED")
print("=" * 60)

HACE COMPONENT CHECK

Expert order:
['fiqa', 'phrasebank', 'twitter', 'general']

✓ ExpertPredictor initialized

✓ Base FeatureFusion initialized
Base feature count: 15
Base features:
 0: fiqa_neg
 1: fiqa_neu
 2: fiqa_pos
 3: phrasebank_neg
 4: phrasebank_neu
 5: phrasebank_pos
 6: twitter_neg
 7: twitter_neu
 8: twitter_pos
 9: general_neg
10: general_neu
11: general_pos
12: token_length
13: prediction_entropy
14: expert_agreement

✓ HACE FeatureFusion initialized
HACE feature count: 18
HACE features:
 0: fiqa_neg
 1: fiqa_neu
 2: fiqa_pos
 3: phrasebank_neg
 4: phrasebank_neu
 5: phrasebank_pos
 6: twitter_neg
 7: twitter_neu
 8: twitter_pos
 9: general_neg
10: general_neu
11: general_pos
12: token_length
13: prediction_entropy
14: expert_agreement
15: hedge_probability
16: hedge_density
17: hedge_count

✓ Base MetaLearner initialized
✓ HACE MetaLearner initialized
✓ HedgeDetector initialized

ALL HACE COMPONENTS INITIALIZED


In [4]:
print("=" * 60)
print("SAVED EXPERTS")
print("=" * 60)

for key in EXPERT_KEYS:

    path = DRIVE_MODEL_DIR / key

    print(
        f"{key:15} : "
        f"{'FOUND' if path.exists() else 'MISSING'}"
    )

SAVED EXPERTS
fiqa            : FOUND
phrasebank      : FOUND
twitter         : FOUND
general         : MISSING


In [6]:
!pip install -q sentencepiece

In [8]:
# ============================================================
# DIAGNOSTIC — FIND WHICH EXPERT FAILS
# ============================================================

from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from src.models.predictor import EXPERT_KEYS
from src.utils.config import config

domain_paths = {
    "fiqa":         DRIVE_MODEL_DIR / "fiqa",
    "phrasebank":   DRIVE_MODEL_DIR / "phrasebank",
    "twitter":      DRIVE_MODEL_DIR / "twitter",
    "finance_news": DRIVE_MODEL_DIR / "finance_news",
    "general":      DRIVE_MODEL_DIR / "general_finbert",
}

for key in EXPERT_KEYS:

    path = domain_paths[key]

    if not path.exists():
        print(f"\n[{key}] DIRECTORY MISSING")
        print(f"→ Testing fallback: {config.finbert_model_name}")
        path = config.finbert_model_name

    print("\n" + "=" * 60)
    print(f"TESTING: {key}")
    print(f"PATH: {path}")
    print("=" * 60)

    try:
        tokenizer = AutoTokenizer.from_pretrained(
            str(path),
            use_fast=False
        )

        print(
            "✓ Tokenizer:",
            type(tokenizer).__name__
        )

        model = AutoModelForSequenceClassification.from_pretrained(
            str(path)
        )

        print("✓ Model loaded")

        del tokenizer
        del model

    except Exception as e:
        print(f"❌ FAILED: {key}")
        print(type(e).__name__)
        print(str(e)[:1000])


TESTING: fiqa
PATH: /content/drive/MyDrive/HACE_models/fiqa
✓ Tokenizer: BertTokenizer


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✓ Model loaded

TESTING: phrasebank
PATH: /content/drive/MyDrive/HACE_models/phrasebank
✓ Tokenizer: BertTokenizer


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✓ Model loaded

TESTING: twitter
PATH: /content/drive/MyDrive/HACE_models/twitter
✓ Tokenizer: BertTokenizer


...

✓ Model loaded

[general] DIRECTORY MISSING
→ Testing fallback: ProsusAI/finbert

TESTING: general
PATH: ProsusAI/finbert


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


✓ Tokenizer: BertTokenizer


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✓ Model loaded


In [9]:
# ============================================================
# CELL 4 — LOAD 4 EXPERTS ONLY
# Finance News intentionally skipped
# ============================================================

from src.models.finbert_expert import FinBERTExpert
from src.utils.config import config

ACTIVE_EXPERT_KEYS = [
    "fiqa",
    "phrasebank",
    "twitter",
    "general"
]

domain_paths = {
    "fiqa":         DRIVE_MODEL_DIR / "fiqa",
    "phrasebank":   DRIVE_MODEL_DIR / "phrasebank",
    "twitter":      DRIVE_MODEL_DIR / "twitter",
    "general":      DRIVE_MODEL_DIR / "general_finbert",
}

experts = {}

for key in ACTIVE_EXPERT_KEYS:

    path = domain_paths[key]

    if not path.exists():

        print(
            f"[ExpertPredictor] {key}: "
            f"loading base FinBERT"
        )

        path = config.finbert_model_name

    print(f"Loading {key}...")

    experts[key] = FinBERTExpert(
        model_path=path,
        domain=key,
        device=DEVICE
    )

    print(f"✓ {key} loaded")

print("\n" + "=" * 60)
print("ACTIVE EXPERTS")
print("=" * 60)

print(ACTIVE_EXPERT_KEYS)

Loading fiqa...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✓ fiqa loaded
Loading phrasebank...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✓ phrasebank loaded
Loading twitter...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✓ twitter loaded
[ExpertPredictor] general: loading base FinBERT
Loading general...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✓ general loaded

ACTIVE EXPERTS
['fiqa', 'phrasebank', 'twitter', 'general']


In [10]:
FINAL_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "final"
)

def load_data(filename):

    path = FINAL_DATA_DIR / filename

    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(path)

    print(
        f"{filename:35} {df.shape}"
    )

    return df


fiqa_train = load_data(
    "fiqa_train.csv"
)

fiqa_val = load_data(
    "fiqa_validation.csv"
)

fiqa_test = load_data(
    "fiqa_test.csv"
)

twitter_train = load_data(
    "twitter_train.csv"
)

twitter_val = load_data(
    "twitter_validation.csv"
)

phrasebank_train = load_data(
    "phrasebank_train.csv"
)

phrasebank_val = load_data(
    "phrasebank_validation.csv"
)

phrasebank_test = load_data(
    "phrasebank_test.csv"
)

finance_train = load_data(
    "finance_news_train.csv"
)

finance_val = load_data(
    "finance_news_validation.csv"
)

finance_test = load_data(
    "finance_news_test.csv"
)

fiqa_train.csv                      (770, 3)
fiqa_validation.csv                 (111, 3)
fiqa_test.csv                       (230, 3)
twitter_train.csv                   (9540, 3)
twitter_validation.csv              (2388, 3)
phrasebank_train.csv                (1807, 3)
phrasebank_validation.csv           (226, 3)
phrasebank_test.csv                 (226, 3)
finance_news_train.csv              (25928, 3)
finance_news_validation.csv         (3241, 3)
finance_news_test.csv               (3241, 3)


In [11]:
meta_train = pd.concat(
    [
        fiqa_val[
            ["text", "label"]
        ].assign(domain="fiqa"),

        twitter_val[
            ["text", "label"]
        ].assign(domain="twitter"),

        phrasebank_val[
            ["text", "label"]
        ].assign(domain="phrasebank"),

        finance_val[
            ["text", "label"]
        ].assign(domain="finance_news"),
    ],
    ignore_index=True
)

meta_train = meta_train.dropna(
    subset=["text", "label"]
)

meta_train["text"] = (
    meta_train["text"]
    .astype(str)
    .str.strip()
)

meta_train = meta_train[
    meta_train["text"] != ""
].reset_index(drop=True)

print("Meta-training:", meta_train.shape)

print("\nDomains:")
print(meta_train["domain"].value_counts())

print("\nLabels:")
print(meta_train["label"].value_counts().sort_index())

Meta-training: (5966, 3)

Domains:
domain
finance_news    3241
twitter         2388
phrasebank       226
fiqa             111
Name: count, dtype: int64

Labels:
label
0    1498
1    1698
2    2770
Name: count, dtype: int64


In [12]:
meta_test = pd.concat(
    [
        fiqa_test[
            ["text", "label"]
        ].assign(domain="fiqa"),

        phrasebank_test[
            ["text", "label"]
        ].assign(domain="phrasebank"),

        finance_test[
            ["text", "label"]
        ].assign(domain="finance_news"),
    ],
    ignore_index=True
)

meta_test = meta_test.dropna(
    subset=["text", "label"]
)

meta_test["text"] = (
    meta_test["text"]
    .astype(str)
    .str.strip()
)

meta_test = meta_test[
    meta_test["text"] != ""
].reset_index(drop=True)

print("Meta-test:", meta_test.shape)

print("\nDomains:")
print(meta_test["domain"].value_counts())

print("\nLabels:")
print(meta_test["label"].value_counts().sort_index())

Meta-test: (3697, 3)

Domains:
domain
finance_news    3241
fiqa             230
phrasebank       226
Name: count, dtype: int64

Labels:
label
0    1196
1    1225
2    1276
Name: count, dtype: int64


In [16]:
# ============================================================
# CELL 9 — GENERATE 4-EXPERT PREDICTIONS
# Finance News skipped temporarily
# ============================================================

print("=" * 60)
print("GENERATING 4-EXPERT PREDICTIONS")
print("=" * 60)

def generate_expert_predictions(df):

    rows = []

    for text in df["text"]:

        row = {}

        for key in ACTIVE_EXPERT_KEYS:

            probabilities = experts[key].predict_proba(
                str(text)
            )

            row[f"{key}_neg"] = float(probabilities[0])
            row[f"{key}_neu"] = float(probabilities[1])
            row[f"{key}_pos"] = float(probabilities[2])

        rows.append(row)

    return pd.DataFrame(
        rows,
        index=df.index
    )


train_expert_df = generate_expert_predictions(
    meta_train
)

test_expert_df = generate_expert_predictions(
    meta_test
)

print("\n✓ Training expert predictions generated")
print("Shape:", train_expert_df.shape)

print("\n✓ Test expert predictions generated")
print("Shape:", test_expert_df.shape)

print("\nColumns:")
print(train_expert_df.columns.tolist())

print("\nFirst 5 rows:")
display(train_expert_df.head())

GENERATING 4-EXPERT PREDICTIONS


KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 10 — FORMAT EXPERT PROBABILITIES
# ============================================================

def dataframe_to_expert_probas(df):

    expert_probas = []

    for _, row in df.iterrows():

        sample = {}

        for key in EXPERT_KEYS:

            sample[key] = [
                float(row[f"{key}_neg"]),
                float(row[f"{key}_neu"]),
                float(row[f"{key}_pos"])
            ]

        expert_probas.append(sample)

    return expert_probas


train_expert_probas = dataframe_to_expert_probas(
    train_expert_df
)

test_expert_probas = dataframe_to_expert_probas(
    test_expert_df
)

print("✓ Expert probabilities formatted")

print("\nExample:")
print(train_expert_probas[0])

In [ ]:
hedge_detector = HedgeDetector()

def generate_hedge_features(texts):

    results = []

    for text in texts:

        result = hedge_detector.detect(
            str(text)
        )

        results.append(result)

    return results


train_hedge_features = generate_hedge_features(
    meta_train["text"].tolist()
)

test_hedge_features = generate_hedge_features(
    meta_test["text"].tolist()
)

print("✓ Training hedging features generated")
print("✓ Test hedging features generated")

print("\nExample:")
print(train_hedge_features[0])

In [ ]:
X_train_base = base_fusion.assemble_batch(
    expert_probas_list=train_expert_probas,
    texts=meta_train["text"].tolist()
)

X_test_base = base_fusion.assemble_batch(
    expert_probas_list=test_expert_probas,
    texts=meta_test["text"].tolist()
)

y_train = (
    meta_train["label"]
    .astype(int)
    .to_numpy()
)

y_test = (
    meta_test["label"]
    .astype(int)
    .to_numpy()
)

print(
    "X_train_base:",
    X_train_base.shape
)

print(
    "X_test_base:",
    X_test_base.shape
)

In [ ]:
X_train_hace = hace_fusion.assemble_batch(
    expert_probas_list=train_expert_probas,
    hedge_features_list=train_hedge_features,
    texts=meta_train["text"].tolist()
)

X_test_hace = hace_fusion.assemble_batch(
    expert_probas_list=test_expert_probas,
    hedge_features_list=test_hedge_features,
    texts=meta_test["text"].tolist()
)

print(
    "X_train_hace:",
    X_train_hace.shape
)

print(
    "X_test_hace:",
    X_test_hace.shape
)

In [ ]:
base_ensemble = MetaLearner(
    use_hedge_features=False,
    C=1.0,
    max_iter=1000,
    seed=SEED
)

base_ensemble.fit(
    X_train_base,
    y_train
)

base_pred = base_ensemble.predict(
    X_test_base
)

base_proba = base_ensemble.predict_proba(
    X_test_base
)

print("✓ Base Ensemble trained")

In [ ]:
hace_model = MetaLearner(
    use_hedge_features=True,
    C=1.0,
    max_iter=1000,
    seed=SEED
)

hace_model.fit(
    X_train_hace,
    y_train
)

hace_pred = hace_model.predict(
    X_test_hace
)

hace_proba = hace_model.predict_proba(
    X_test_hace
)

print("✓ HACE trained")

In [ ]:
def evaluate_model(
    name,
    y_true,
    y_pred
):

    return {
        "Model": name,

        "Accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "Macro Precision":
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "Macro Recall":
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "Macro F1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "Weighted F1":
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0
            )
    }


results = pd.DataFrame([
    evaluate_model(
        "Base Ensemble",
        y_test,
        base_pred
    ),

    evaluate_model(
        "HACE",
        y_test,
        hace_pred
    )
])

print(
    results.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

In [ ]:
print("=" * 60)
print("HACE CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_test,
        hace_pred,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ],
        digits=4,
        zero_division=0
    )
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        hace_pred
    )
)

In [ ]:
comparison = []

for key in EXPERT_KEYS:

    probs = np.array([
        [
            row[key][0],
            row[key][1],
            row[key][2]
        ]
        for row in test_expert_probas
    ])

    pred = np.argmax(
        probs,
        axis=1
    )

    comparison.append(
        evaluate_model(
            key,
            y_test,
            pred
        )
    )


comparison.append(
    evaluate_model(
        "Base Ensemble",
        y_test,
        base_pred
    )
)

comparison.append(
    evaluate_model(
        "HACE",
        y_test,
        hace_pred
    )
)

comparison_df = pd.DataFrame(
    comparison
).sort_values(
    "Macro F1",
    ascending=False
)

print(
    comparison_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

In [ ]:
base_f1 = f1_score(
    y_test,
    base_pred,
    average="macro"
)

hace_f1 = f1_score(
    y_test,
    hace_pred,
    average="macro"
)

absolute_gain = (
    hace_f1 - base_f1
)

print("=" * 60)
print("HACE ABLATION STUDY")
print("=" * 60)

print(
    f"Base Ensemble Macro F1 : {base_f1:.4f}"
)

print(
    f"HACE Macro F1          : {hace_f1:.4f}"
)

print(
    f"Gain from Hedging      : {absolute_gain:+.4f}"
)

In [ ]:
MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "meta_learner"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

base_path = base_ensemble.save(
    MODEL_DIR
)

hace_path = hace_model.save(
    MODEL_DIR
)

print("Base:", base_path)
print("HACE:", hace_path)

In [ ]:
import shutil

DRIVE_META_DIR = (
    DRIVE_MODEL_DIR
    / "meta_learner"
)

DRIVE_META_DIR.mkdir(
    parents=True,
    exist_ok=True
)

shutil.copy2(
    base_path,
    DRIVE_META_DIR
    / "base_ensemble.pkl"
)

shutil.copy2(
    hace_path,
    DRIVE_META_DIR
    / "hace_meta_learner.pkl"
)

print(
    "✓ Saved to:",
    DRIVE_META_DIR
)

In [ ]:
OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

comparison_df.to_csv(
    OUTPUT_DIR
    / "expert_vs_ensemble.csv",
    index=False
)

results.to_csv(
    OUTPUT_DIR
    / "base_vs_hace.csv",
    index=False
)

ablation = {
    "base_ensemble_macro_f1":
        float(base_f1),

    "hace_macro_f1":
        float(hace_f1),

    "hedging_gain":
        float(absolute_gain)
}

import json

with open(
    OUTPUT_DIR
    / "hace_ablation.json",
    "w"
) as f:

    json.dump(
        ablation,
        f,
        indent=4
    )

print("✓ Results saved")